# C5: Conditional Graph Deadlocks in DDP

## The Core Issue
When using Zygote for automatic differentiation, if a parameter is not used in the forward pass of a specific rank (e.g. due to a conditional branch `if rank == 0 ...`), Zygote returns `nothing` for that parameter's gradient.

Optimisers.jl handles `nothing` gradients by simply skipping the update for that parameter. However, in Distributed Data Parallel (DDP) training, `DistributedOptimizer` wraps the `apply!` function to perform an MPI `allreduce!` on the gradients before the update.

If rank A returns an `Array` for a gradient, it enters the `allreduce!` collective. If rank B returns `nothing` for the same gradient, Optimisers.jl skips it, and rank B *never enters* the collective. Rank A waits forever at the barrier, causing a deadlock.

## The Fix
To solve this, we introduce `DistributedUtils.resolve_unused_parameters!(backend, gs, model)`. This function traverses the gradient tree (`gs`) and the model tree simultaneously. If it encounters a `nothing` in the gradient tree corresponding to an `AbstractArray` in the model tree, it substitutes the `nothing` with a zero-filled array on the same device and with the same type as the model parameter.

This ensures that all ranks participate in the `allreduce!` collective for every parameter, regardless of whether the parameter was used in the local forward pass. If a parameter was unused locally, its contribution to the global gradient will be zero, which is mathematically correct.
